# Pt(100) bulk DWBA crystal truncation rods in the Vlieg z-mode

This example uses `SXRDCrystal.prepare_DWBA_Zmode`, which derives the
incident and exit glancing angles from the z-mode angle constraint of
`HKLVlieg.VliegAngles.anglesZmode` instead of requiring explicit
angles.

Two rods are calculated for the `Pt100` bulk cell:

* the specular rod `(0 0 L)` with `fixed="eq"`, so that the incidence
  and exit angles are equal and both grow with `L`,
* the non-specular rod `(2 0 L)` with `fixed="in"` at a fixed
  incidence angle of 0.2 deg, which is just below the critical angle
  of Pt at 20 keV.

The distorted-wave amplitudes use the exact absorption contained in
their complex internal wavevectors. They are compared with the
kinematical semi-infinite bulk amplitude of the same model. On the specular rod
the two converge once both glancing angles are well above the
critical angle. On the non-specular rod the incidence angle stays
below the critical angle for every `L`, so the incident wave remains
evanescent and the two models differ over the whole rod.

In [ ]:
%matplotlib widget
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root():
    """Find the checkout containing the current example notebook."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "orgui" / "datautils").is_dir():
            return candidate
    return None


repository_root = find_repository_root()
if repository_root is not None:
    sys.path.insert(0, str(repository_root))

from orgui.datautils.xrayutils import CTRcalc, CTRutil, unitcells
from orgui.datautils.xrayutils.CTRoptics import homogeneous_bulk_profile

if (
    repository_root is not None
    and repository_root not in Path(CTRcalc.__file__).resolve().parents
):
    raise RuntimeError(
        "An installed orgui version is already loaded. Restart the "
        "kernel and run the notebook from the first cell."
    )

## Model and critical angle

DWBA needs no empirical attenuation: absorption is already contained
in its complex internal wavevectors. The kinematical model is first
shown with zero attenuation. On the fixed-incidence `(2 0 L)` rod it
is also evaluated with the incident-path attenuation obtained from
`CTRutil.set_atten_from_dwba`. The critical angle follows from the
dispersion of the homogeneous bulk medium that the DWBA optics see,
$\alpha_c=\sqrt{2\delta}$.

In [ ]:
#: Photon energy in eV.
ENERGY_EV = 20000.0
#: Fixed incidence angle of the non-specular rod in degrees.
NONSPECULAR_ALPHA_I_DEG = 0.2

uc_bulk = unitcells.unitcell("Pt100")
uc_bulk.setEnergy(ENERGY_EV)
crystal = CTRcalc.SXRDCrystal(uc_bulk, atten=0.0)

profile = homogeneous_bulk_profile(uc_bulk)
dispersion = float(np.max(profile.values[:, 1]))
alpha_c = float(np.sqrt(2.0 * dispersion))
print(
    f"Pt at {ENERGY_EV * 1e-3:g} keV: delta = {dispersion:.4g}, "
    f"critical angle = {np.rad2deg(alpha_c):.4f} deg"
)
uc_bulk

## Rod evaluation

`prepare_DWBA_Zmode` populates the diffractometer from the crystal
itself: the bulk cell is the lattice, its energy sets the wavelength,
`h`, `k`, and `l` are in reference-cell r.l.u., and the orientation
matrix is `U = 1`, so the surface normal is the omega axis. It returns
the immutable `PreparedDWBA` snapshot, and with `return_angles=True`
also the six z-mode angles `(alpha, delta, gamma, omega, chi, phi)` in
radians.

The zero-attenuation kinematical reference is `UnitCell.F_bulk`. For a
fixed incidence angle, `CTRutil.set_atten_from_dwba` can populate its
scalar empirical attenuation from the exact incident DWBA wavevector.

In [ ]:
def rod(crystal, h, k, L, fixedangle, fixed):
    """Evaluate one rod in the DWBA and in the kinematical approximation.

    :param SXRDCrystal crystal:
        Bulk-only crystal model.
    :param float h:
        Reference-frame in-plane index in r.l.u.
    :param float k:
        Reference-frame in-plane index in r.l.u.
    :param numpy.ndarray L:
        Reference-frame out-of-plane index in r.l.u.
    :param float fixedangle:
        Value of the z-mode fixed angle in radians.
    :param str fixed:
        z-mode angle constraint, ``"in"``, ``"out"`` or ``"eq"``.
    :returns:
        DWBA amplitude, kinematical amplitude, incidence angle in
        radians and exit angle in radians.
    :rtype: tuple
    """
    prepared, angles = crystal.prepare_DWBA_Zmode(
        h, k, L, fixedangle, fixed=fixed, return_angles=True
    )
    F_dwba = crystal.F_DWBA_prepared(prepared)
    h_array = np.full_like(L, float(h))
    k_array = np.full_like(L, float(k))
    F_kinematical = crystal.uc_bulk.F_bulk(
        h_array, k_array, L, 0.0
    )
    return F_dwba, F_kinematical, angles[:, 0], angles[:, 2]

### Specular rod `(0 0 L)`, `fixed="eq"`

The equal-angle constraint splits the out-of-plane momentum transfer
evenly between the incident and exit wave, so `fixedangle` has no
effect and both angles grow with `L`.

In [ ]:
L_specular = np.linspace(0.0001, 4.0, 2001)
F_dwba_s, F_kin_s, alpha_i_s, alpha_f_s = rod(
    crystal, 0.0, 0.0, L_specular, 0.0, "eq"
)

np.testing.assert_allclose(alpha_i_s, alpha_f_s)
print(
    f"alpha at L = {L_specular[0]:.2f}: "
    f"{np.rad2deg(alpha_i_s[0]):.4f} deg\n"
    f"alpha at L = {L_specular[-1]:.2f}: "
    f"{np.rad2deg(alpha_i_s[-1]):.4f} deg"
)

### Non-specular rod `(2 0 L)`, `fixed="in"`

The incidence angle is held at 0.2 deg, below the critical angle, and
the exit angle carries the whole out-of-plane momentum transfer.
Above `L` of about 4.6 the reflection leaves the Ewald sphere at this
incidence angle and energy, and `prepare_DWBA_Zmode` rejects it.

In [ ]:
L_rod = np.linspace(0.025, 4.0, 2001)
alpha_i_fixed = np.deg2rad(NONSPECULAR_ALPHA_I_DEG)
F_dwba_r, F_kin_r, alpha_i_r, alpha_f_r = rod(
    crystal, 2.0, 0.0, L_rod, alpha_i_fixed, "in"
)
atten_incident = CTRutil.set_atten_from_dwba(crystal, alpha_i_fixed)
F_kin_r_attenuated = crystal.uc_bulk.F_bulk(
    np.full_like(L_rod, 2.0), np.zeros_like(L_rod), L_rod, crystal.atten
)

print(
    f"alpha_i = {np.rad2deg(alpha_i_r[0]):.4f} deg (fixed)\n"
    f"alpha_f from {np.rad2deg(alpha_f_r[0]):.4f} deg "
    f"to {np.rad2deg(alpha_f_r[-1]):.4f} deg\n"
    f"incident-path kinematical atten = {atten_incident:.6f}"
)

try:
    crystal.prepare_DWBA_Zmode(
        2.0,
        0.0,
        5.0,
        np.deg2rad(NONSPECULAR_ALPHA_I_DEG),
        fixed="in",
    )
except ValueError as error:
    print(f"\nL = 5.0 is rejected: {error}")

## Rods and z-mode angles

In [ ]:
def plot_rod(axes, L, F_dwba, F_kinematical, title,
             F_kinematical_attenuated=None):
    """Plot the DWBA and kinematical intensity of one rod."""
    axes.semilogy(
        L, np.abs(F_kinematical) ** 2, "--", color="0.5",
        label="kinematical, atten=0",
    )
    if F_kinematical_attenuated is not None:
        axes.semilogy(
            L, np.abs(F_kinematical_attenuated) ** 2, "-.", color="C3",
            label=rf"kinematical, atten={crystal.atten:.4f}",
        )
    axes.semilogy(L, np.abs(F_dwba) ** 2, "-", color="C0", label="DWBA")
    axes.set_xlabel("L / r.l.u.")
    axes.set_ylabel(r"$|F|^2$ / electrons$^2$")
    axes.set_title(title)
    axes.legend()


def plot_angles(axes, L, alpha_i, alpha_f, alpha_c, title):
    """Plot the z-mode glancing angles against the critical angle."""
    axes.semilogy(
        L, np.rad2deg(alpha_i), "-", color="C1", label=r"$\alpha_i$"
    )
    axes.semilogy(
        L, np.rad2deg(alpha_f), "--", color="C2", label=r"$\alpha_f$"
    )
    axes.axhline(
        np.rad2deg(alpha_c),
        color="0.3",
        linestyle=":",
        label=r"$\alpha_c$",
    )
    axes.set_xlabel("L / r.l.u.")
    axes.set_ylabel("angle / deg")
    axes.set_title(title)
    axes.legend()


figure, axes = plt.subplots(2, 2, figsize=(11.0, 7.5))
plot_rod(
    axes[0, 0], L_specular, F_dwba_s, F_kin_s,
    "specular (0 0 L), fixed='eq'",
)
plot_angles(
    axes[0, 1], L_specular, alpha_i_s, alpha_f_s, alpha_c,
    "(0 0 L) z-mode angles",
)
plot_rod(
    axes[1, 0], L_rod, F_dwba_r, F_kin_r,
    "(2 0 L), fixed='in', "
    rf"$\alpha_i$ = {NONSPECULAR_ALPHA_I_DEG:g}$\degree$",
    F_kinematical_attenuated=F_kin_r_attenuated,
)
plot_angles(
    axes[1, 1], L_rod, alpha_i_r, alpha_f_r, alpha_c,
    "(2 0 L) z-mode angles",
)
figure.suptitle(
    f"Pt(100) bulk DWBA crystal truncation rods at "
    f"{ENERGY_EV * 1e-3:g} keV"
)
figure.tight_layout()

The specular DWBA rod follows the kinematical one down to about
`L = 0.3` and rises above it below that, where the glancing angle
approaches the critical angle. The odd internal reflections are the
systematic fcc absences of the four-atom `Pt100` cell, but truncation
leaves finite anti-Bragg intensity between the allowed bulk peaks.

The non-specular DWBA rod sits above the kinematical one over its
whole length: the incidence angle never leaves the total-reflection
region, so the transmitted incident field, not the vacuum field, is
the one that scatters. The incident-path attenuation improves the
kinematical bulk envelope once the exit attenuation becomes negligible,
but it cannot reproduce the DWBA transmission amplitudes.

## The sub-critical angle range

Everything interesting about the distorted waves happens where a
glancing angle crosses the critical angle, and on an `L` axis that
region is a sliver: the specular rod is sub-critical only for
`L < 0.053`, and the non-specular rod above has its whole sub-critical
exit range in `0.022 < L < 0.049`, entirely below the `L = 0.05` at
which the rods above start.

The angle, not `L`, is the natural scan variable there, so use
`prepare_DWBA_from_angles`. It is the counterpart of
`prepare_DWBA_Zmode`: you give the in-plane indices and both glancing
angles, and it derives `L` from the Ewald condition
$Q_z = k_0(\sin\alpha_i + \sin\alpha_f)$ with
`l_from_glancing_angles`, which is a linear solve because
`B_mat @ refHKLTransform` maps reference r.l.u. to cartesian
$\mathbf{Q}$. It is the same rod as above, only reparameterized.

In [ ]:
def rod_vs_angle(crystal, h, k, alpha_i, alpha_f):
    """Evaluate one rod at given glancing angles, deriving L from them.

    :param SXRDCrystal crystal:
        Bulk-only crystal model.
    :param float h:
        Reference-frame in-plane index in r.l.u.
    :param float k:
        Reference-frame in-plane index in r.l.u.
    :param float or numpy.ndarray alpha_i:
        Incident glancing angle in radians.
    :param float or numpy.ndarray alpha_f:
        Exit glancing angle in radians.
    :returns:
        DWBA amplitude, kinematical amplitude and the derived
        out-of-plane index.
    :rtype: tuple
    """
    prepared, L = crystal.prepare_DWBA_from_angles(
        h, k, alpha_i, alpha_f, return_l=True
    )
    F_dwba = crystal.F_DWBA_prepared(prepared)
    h_array = np.full_like(L, float(h))
    k_array = np.full_like(L, float(k))
    F_kinematical = crystal.uc_bulk.F_bulk(
        h_array, k_array, L, 0.0
    )
    return F_dwba, F_kinematical, L


# Exclude the exact zero: a vanishing glancing angle has no DWBA
# solution, and prepare_DWBA requires both angles in (0, pi/2].
angle_scan = np.linspace(0.0, 5.0 * alpha_c, 601)[1:]
alpha_i_fixed = np.deg2rad(NONSPECULAR_ALPHA_I_DEG)

F_dwba_sa, F_kin_sa, L_specular_scan = rod_vs_angle(
    crystal, 0.0, 0.0, angle_scan, angle_scan
)
F_dwba_ra, F_kin_ra, L_rod_scan = rod_vs_angle(
    crystal, 2.0, 0.0, alpha_i_fixed, angle_scan
)

# The two convenience APIs are exact inverses of each other: feeding the
# derived L back through the z-mode constraint returns the scanned angle.
_, angles_back = crystal.prepare_DWBA_Zmode(
    2.0, 0.0, L_rod_scan, alpha_i_fixed, fixed="in", return_angles=True
)
np.testing.assert_allclose(angles_back[:, 2], angle_scan, atol=1e-12)

print(
    f"critical angle: {np.rad2deg(alpha_c):.4f} deg\n"
    f"scan range: {np.rad2deg(angle_scan[0]):.4f} to "
    f"{np.rad2deg(angle_scan[-1]):.4f} deg\n"
    f"specular L range: {L_specular_scan[0]:.5f} to "
    f"{L_specular_scan[-1]:.5f}\n"
    f"(2 0 L) L range: {L_rod_scan[0]:.5f} to {L_rod_scan[-1]:.5f}"
)

The two panels below scan the glancing angle from 0 to five times the
critical angle. The dashed vertical line is $\alpha_c$.

In [ ]:
def plot_angle_scan(axes, angle, F_dwba, F_kinematical, alpha_c,
                    xlabel, title):
    """Plot DWBA and kinematical intensity against a glancing angle."""
    degrees = np.rad2deg(angle)
    axes.semilogy(
        degrees, np.abs(F_kinematical) ** 2, "--", color="0.5",
        label="kinematical, atten=0",
    )
    axes.semilogy(
        degrees, np.abs(F_dwba) ** 2, "-", color="C0", label="DWBA"
    )
    axes.axvline(
        np.rad2deg(alpha_c),
        color="0.3",
        linestyle="--",
        label=r"$\alpha_c$",
    )
    axes.set_xlabel(xlabel)
    axes.set_ylabel(r"$|F|^2$ / electrons$^2$")
    axes.set_xlim(0.0, degrees[-1])
    axes.set_title(title)
    axes.legend()


figure, axes = plt.subplots(1, 2, figsize=(11.0, 4.2))
plot_angle_scan(
    axes[0], angle_scan, F_dwba_sa, F_kin_sa, alpha_c,
    r"$\alpha$ / deg",
    "specular (0 0 L), fixed='eq'",
)
plot_angle_scan(
    axes[1], angle_scan, F_dwba_ra, F_kin_ra, alpha_c,
    r"$\gamma$ / deg",
    "(2 0 L), fixed='in', "
    rf"$\alpha_i$ = {NONSPECULAR_ALPHA_I_DEG:g}$\degree$",
)
figure.suptitle(
    "Pt(100) bulk DWBA across the critical angle at "
    f"{ENERGY_EV * 1e-3:g} keV"
)
figure.tight_layout()

Both scans peak at the critical angle, where the transmitted field
amplitude into the substrate is largest, and fall back onto the
kinematical curve well above it. The kinematical amplitude knows
nothing about $\alpha_c$ and varies smoothly through it.

On the specular rod the incident and exit wave cross $\alpha_c$
together, so the enhancement is the transmission function squared.
On the non-specular rod the incidence angle is held below
$\alpha_c$ throughout, so only the exit wave crosses it and the
peak is the Yoneda wing of that rod; the DWBA curve therefore stays
above the kinematical one even at the right edge of the scan.

## DWBA-equivalent kinematical attenuation

For fixed incidence, `set_atten_from_dwba` stores the incident-path
attenuation in `crystal.atten`. The full depth decay also contains the
exit path and therefore depends on $\alpha_f$. The incident-only scalar
becomes a good approximation once that exit contribution is small. The
shaded range below uses a 5% relative criterion.

In [ ]:
alpha_f_attenuation = np.geomspace(
    np.deg2rad(0.01), alpha_f_r[-1], 800
)
atten_plateau = CTRutil.set_atten_from_dwba(crystal, alpha_i_fixed)
atten_exact = CTRutil.attenuation_from_dwba(
    crystal, alpha_i_fixed, alpha_f_attenuation
)
L_attenuation = crystal.l_from_glancing_angles(
    2.0, 0.0, alpha_i_fixed, alpha_f_attenuation
)
relative_exit_attenuation = (atten_exact - atten_plateau) / atten_plateau
good_approximation = relative_exit_attenuation <= 0.05
first_good = np.flatnonzero(good_approximation)[0]
alpha_f_good = alpha_f_attenuation[first_good]
L_good = L_attenuation[first_good]

figure, axes = plt.subplots(figsize=(8.0, 4.8))
alpha_f_degrees = np.rad2deg(alpha_f_attenuation)
axes.semilogx(
    alpha_f_degrees, atten_exact, color="C0",
    label=r"exact $\eta_i + \eta_f$ from DWBA",
)
axes.axhline(
    atten_plateau, color="C3", linestyle="--",
    label=rf"`set_atten_from_dwba`: $\eta_i={atten_plateau:.4f}$",
)
axes.axvline(
    np.rad2deg(alpha_c), color="0.35", linestyle=":",
    label=r"$\alpha_c$",
)
axes.axvspan(
    np.rad2deg(alpha_f_good), alpha_f_degrees[-1],
    color="C2", alpha=0.12, label="incident-only error <= 5%",
)
axes.set_xlabel(r"$\alpha_f$ / deg")
axes.set_ylabel("attenuation exponent per reference repeat")
axes.set_xlim(alpha_f_degrees[0], alpha_f_degrees[-1])
axes.set_title(
    rf"Pt(100), fixed $\alpha_i={NONSPECULAR_ALPHA_I_DEG:g}\degree$"
)
axes.legend()

L_axis = axes.twiny()
L_axis.set_xscale(axes.get_xscale())
L_axis.set_xlim(axes.get_xlim())
angle_ticks = np.geomspace(
    alpha_f_attenuation[0], alpha_f_attenuation[-1], 7
)
L_ticks = crystal.l_from_glancing_angles(
    2.0, 0.0, alpha_i_fixed, angle_ticks
)
L_axis.set_xticks(np.rad2deg(angle_ticks))
L_axis.set_xticklabels([f"{value:.3g}" for value in L_ticks])
L_axis.set_xlabel(r"corresponding $L$ / r.l.u. on $(2\ 0\ L)$")
figure.tight_layout()

print(
    f"incident-only atten = {atten_plateau:.6f}\n"
    f"within 5% for alpha_f >= {np.rad2deg(alpha_f_good):.4f} deg "
    f"(L >= {L_good:.5f})"
)